In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture

import warnings
warnings.filterwarnings('ignore')

In [2]:
transaction_data = pd.read_csv("../dunnhumby_The-Complete-Journey/transaction_data.csv")
campaign_table = pd.read_csv("../dunnhumby_The-Complete-Journey/campaign_table.csv")
campaign_desc = pd.read_csv("../dunnhumby_The-Complete-Journey/campaign_desc.csv")
coupon = pd.read_csv("../dunnhumby_The-Complete-Journey/coupon.csv")
coupon_redempt = pd.read_csv("../dunnhumby_The-Complete-Journey/coupon_redempt.csv")
product = pd.read_csv("../dunnhumby_The-Complete-Journey/product.csv")
segmented_data = pd.read_csv("../dunnhumby_The-Complete-Journey/segmented_df.csv")

In [3]:
all_households = transaction_data["household_key"].unique()
len(all_households)

2500

In [4]:
# loading data
all_weeks = range(
    transaction_data["WEEK_NO"].min(),
    transaction_data["WEEK_NO"].max() + 1
)

full_panel = pd.MultiIndex.from_product(
    [all_households, all_weeks],
    names=["household_key", "WEEK_NO"]
).to_frame(index=False)

In [5]:
pd.set_option('display.max_rows',None)

In [6]:
full_panel.head()

,household_key,WEEK_NO
0,2375,1
1,2375,2
2,2375,3
3,2375,4
4,2375,5


In [7]:
full_panel.shape

(255000, 2)

In [8]:
weekly_sales = (
    transaction_data
    .groupby(['household_key', 'WEEK_NO'])
    .agg(
        sales=('SALES_VALUE', 'sum'),
        quantity=('QUANTITY', 'sum'),
        baskets=('BASKET_ID', 'nunique'),
        unique_products=('PRODUCT_ID', 'nunique')
    )
    .reset_index()
)

In [9]:
panel = full_panel.merge(
    weekly_sales,
    on=["household_key", "WEEK_NO"],
    how="left"
)

In [10]:
panel.shape

(255000, 6)

In [11]:
panel.head()

,household_key,WEEK_NO,sales,quantity,baskets,unique_products
0,2375,1,18.34,12.0,2.0,11.0
1,2375,2,169.04,44.0,1.0,31.0
2,2375,3,NaN,NaN,NaN,NaN
3,2375,4,NaN,NaN,NaN,NaN
4,2375,5,NaN,NaN,NaN,NaN


In [12]:
panel.isnull().sum()

household_key           0
WEEK_NO                 0
sales              131024
quantity           131024
baskets            131024
unique_products    131024
dtype: int64

In [13]:
# weekly panel for sales
panel["sales"] = panel["sales"].fillna(0)
panel["quantity"] = panel["quantity"].fillna(0)

panel["baskets"] = panel["baskets"].fillna(0)
panel["unique_products"] = panel["unique_products"].fillna(0)

In [14]:
panel.groupby("household_key").size().describe()


count    2500.0
mean      102.0
std         0.0
min       102.0
25%       102.0
50%       102.0
75%       102.0
max       102.0
dtype: float64

In [15]:
campaign_desc['start_week'] = (campaign_desc['START_DAY'] // 7) + 1

campaign_desc['end_week'] = (campaign_desc['END_DAY'] // 7) + 1

In [16]:
campaign_info = campaign_table.merge(
    campaign_desc,
    on=['CAMPAIGN', 'DESCRIPTION'],
    how='left'
)

In [17]:
campaign_info.head()

,DESCRIPTION,household_key,CAMPAIGN,START_DAY,END_DAY,start_week,end_week
0,TypeA,17,26,224,264,33,38
1,TypeA,27,26,224,264,33,38
2,TypeA,212,26,224,264,33,38
3,TypeA,208,26,224,264,33,38
4,TypeA,192,26,224,264,33,38


In [18]:
rows = []
for _, row in campaign_info.iterrows():
    print(row)

DESCRIPTION      TypeA
household_key       17
CAMPAIGN            26
START_DAY          224
END_DAY            264
start_week          33
end_week            38
Name: 0, dtype: object
DESCRIPTION      TypeA
household_key       27
CAMPAIGN            26
START_DAY          224
END_DAY            264
start_week          33
end_week            38
Name: 1, dtype: object
DESCRIPTION      TypeA
household_key      212
CAMPAIGN            26
START_DAY          224
END_DAY            264
start_week          33
end_week            38
Name: 2, dtype: object
DESCRIPTION      TypeA
household_key      208
CAMPAIGN            26
START_DAY          224
END_DAY            264
start_week          33
end_week            38
Name: 3, dtype: object
DESCRIPTION      TypeA
household_key      192
CAMPAIGN            26
START_DAY          224
END_DAY            264
start_week          33
end_week            38
Name: 4, dtype: object
DESCRIPTION      TypeA
household_key      187
CAMPAIGN            26
START_DAY  

In [19]:
rows = []
for _, row in campaign_info.iterrows():
    for week in range(row['start_week'], row['end_week'] + 1):
        rows.append({
            'household_key': row['household_key'],
            'WEEK_NO': week,
            'campaign_type': row['DESCRIPTION']
        })

campaign_weeks = pd.DataFrame(rows)

In [20]:
campaign_weeks.head()

,household_key,WEEK_NO,campaign_type
0,17,33,TypeA
1,17,34,TypeA
2,17,35,TypeA
3,17,36,TypeA
4,17,37,TypeA


In [21]:
campaign_desc['DESCRIPTION'].value_counts()

DESCRIPTION
TypeB    19
TypeC     6
TypeA     5
Name: count, dtype: int64

In [22]:
campaign_dummies = pd.get_dummies(campaign_weeks,columns=['campaign_type'], dtype=int)
campaign_dummies.head()

,household_key,WEEK_NO,campaign_type_TypeA,campaign_type_TypeB,campaign_type_TypeC
0,17,33,1,0,0
1,17,34,1,0,0
2,17,35,1,0,0
3,17,36,1,0,0
4,17,37,1,0,0


In [23]:
panel = panel.merge(
    campaign_dummies,
    on=["household_key", "WEEK_NO"],
    how="left"
)

In [24]:
pd.set_option('display.max_columns', None)

In [25]:
panel.isnull().sum()

household_key               0
WEEK_NO                     0
sales                       0
quantity                    0
baskets                     0
unique_products             0
campaign_type_TypeA    213221
campaign_type_TypeB    213221
campaign_type_TypeC    213221
dtype: int64

In [26]:
campaign_cols = [
    "campaign_type_TypeA",
    "campaign_type_TypeB",
    "campaign_type_TypeC"
]

panel[campaign_cols] = (
    panel[campaign_cols]
    .fillna(0)
)


In [27]:
panel = panel.sort_values(
    ["household_key", "WEEK_NO"]
)

In [28]:
# adding lag features
for col in campaign_cols:

    panel[f"{col}_lag4"] = (
        panel
        .groupby("household_key")[col]
        .shift(4)
        .fillna(0)
    )

In [29]:
panel["sales_lag1"] = (
    panel
    .groupby("household_key")["sales"]
    .shift(1)
)

In [30]:
panel["sales_lag4"] = (
    panel
    .groupby("household_key")["sales"]
    .shift(4)
)

In [31]:
panel["qty_lag1"] = (
    panel
    .groupby("household_key")["quantity"]
    .shift(1)
)

panel["qty_lag4"] = (
    panel
    .groupby("household_key")["quantity"]
    .shift(4)
)

In [32]:
lag_cols = [
    "sales_lag1",
    "sales_lag4",
    "qty_lag1",
    "qty_lag4"
]

In [33]:
panel[lag_cols] = (
    panel[lag_cols]
    .fillna(0)
)

In [34]:
panel = panel.merge(segmented_data, on='household_key', how='left')

In [35]:
segment_dummies = pd.get_dummies(panel['cluster'], prefix='segment', dtype=int)

In [36]:
panel = pd.concat(
    [panel, segment_dummies],
    axis=1
)

In [37]:
# fixed effects panel regression
segment_cols = segment_dummies.columns

for campaign in [
    "campaign_type_TypeA_lag4",
    "campaign_type_TypeB_lag4",
    "campaign_type_TypeC_lag4"
]:

    for segment in segment_cols:

        panel[f"{campaign}_X_{segment}"] = (
            panel[campaign]
            * panel[segment]
        )


In [38]:
panel = panel.set_index(
    ["household_key", "WEEK_NO"]
)


In [39]:
panel.head()

sales  quantity  baskets  unique_products  \
household_key WEEK_NO                                              
1             1          0.0       0.0      0.0              0.0   
              2          0.0       0.0      0.0              0.0   
              3          0.0       0.0      0.0              0.0   
              4          0.0       0.0      0.0              0.0   
              5          0.0       0.0      0.0              0.0   

                       campaign_type_TypeA  campaign_type_TypeB  \
household_key WEEK_NO                                             
1             1                        0.0                  0.0   
              2                        0.0                  0.0   
              3                        0.0                  0.0   
              4                        0.0                  0.0   
              5                        0.0                  0.0   

                       campaign_type_TypeC  campaign_type_TypeA_lag4  \
household_key WEEK_NO                                                  
1             1                        0.0                       0.0   
              2                        0.0                       0.0   
              3                        0.0                       0.0   
              4                        0.0                       0.0   
              5                        0.0                       0.0   

                       campaign_type_TypeB_lag4  campaign_type_TypeC_lag4  \
household_key WEEK_NO                                                       
1             1                             0.0                       0.0   
              2                             0.0                       0.0   
              3                             0.0                       0.0   
              4                             0.0                       0.0   
              5                             0.0                       0.0   

                       sales_lag1  sales_lag4  qty_lag1  qty_lag4  cluster  \
household_key WEEK_NO                                                        
1             1               0.0         0.0       0.0       0.0        1   
              2               0.0         0.0       0.0       0.0        1   
              3               0.0         0.0       0.0       0.0        1   
              4               0.0         0.0       0.0       0.0        1   
              5               0.0         0.0       0.0       0.0        1   

                       segment_0  segment_1  segment_2  \
household_key WEEK_NO                                    
1             1                0          1          0   
              2                0          1          0   
              3                0          1          0   
              4                0          1          0   
              5                0          1          0   

                       campaign_type_TypeA_lag4_X_segment_0  \
household_key WEEK_NO                                         
1             1                                         0.0   
              2                                         0.0   
              3                                         0.0   
              4                                         0.0   
              5                                         0.0   

                       campaign_type_TypeA_lag4_X_segment_1  \
household_key WEEK_NO                                         
1             1                                         0.0   
              2                                         0.0   
              3                                         0.0   
              4                                         0.0   
              5                                         0.0   

                       campaign_type_TypeA_lag4_X_segment_2  \
household_key WEEK_NO                                         
1             1                                         0.0   
    

In [40]:
base_features_sales = [
    "campaign_type_TypeA_lag4",
    "campaign_type_TypeB_lag4",
    "campaign_type_TypeC_lag4",

    "sales_lag1",
    "sales_lag4",

    "baskets",
    "unique_products"
]


In [41]:
base_features_qty = [
    "campaign_type_TypeA_lag4",
    "campaign_type_TypeB_lag4",
    "campaign_type_TypeC_lag4",

    "qty_lag1",
    "qty_lag4",

    "baskets",
    "unique_products"
]


In [42]:
interaction_cols = [
    col
    for col in panel.columns
    if "_X_segment_" in col
]


In [43]:
interaction_cols

['campaign_type_TypeA_lag4_X_segment_0',
 'campaign_type_TypeA_lag4_X_segment_1',
 'campaign_type_TypeA_lag4_X_segment_2',
 'campaign_type_TypeB_lag4_X_segment_0',
 'campaign_type_TypeB_lag4_X_segment_1',
 'campaign_type_TypeB_lag4_X_segment_2',
 'campaign_type_TypeC_lag4_X_segment_0',
 'campaign_type_TypeC_lag4_X_segment_1',
 'campaign_type_TypeC_lag4_X_segment_2']

In [44]:
X_sales = panel[
    base_features_sales + interaction_cols
]

In [45]:
y_sales = panel["sales"]

In [46]:
X_qty = panel[
    base_features_qty + interaction_cols
]

In [47]:
y_qty = panel["quantity"]

In [48]:
panel.head()

sales  quantity  baskets  unique_products  \
household_key WEEK_NO                                              
1             1          0.0       0.0      0.0              0.0   
              2          0.0       0.0      0.0              0.0   
              3          0.0       0.0      0.0              0.0   
              4          0.0       0.0      0.0              0.0   
              5          0.0       0.0      0.0              0.0   

                       campaign_type_TypeA  campaign_type_TypeB  \
household_key WEEK_NO                                             
1             1                        0.0                  0.0   
              2                        0.0                  0.0   
              3                        0.0                  0.0   
              4                        0.0                  0.0   
              5                        0.0                  0.0   

                       campaign_type_TypeC  campaign_type_TypeA_lag4  \
household_key WEEK_NO                                                  
1             1                        0.0                       0.0   
              2                        0.0                       0.0   
              3                        0.0                       0.0   
              4                        0.0                       0.0   
              5                        0.0                       0.0   

                       campaign_type_TypeB_lag4  campaign_type_TypeC_lag4  \
household_key WEEK_NO                                                       
1             1                             0.0                       0.0   
              2                             0.0                       0.0   
              3                             0.0                       0.0   
              4                             0.0                       0.0   
              5                             0.0                       0.0   

                       sales_lag1  sales_lag4  qty_lag1  qty_lag4  cluster  \
household_key WEEK_NO                                                        
1             1               0.0         0.0       0.0       0.0        1   
              2               0.0         0.0       0.0       0.0        1   
              3               0.0         0.0       0.0       0.0        1   
              4               0.0         0.0       0.0       0.0        1   
              5               0.0         0.0       0.0       0.0        1   

                       segment_0  segment_1  segment_2  \
household_key WEEK_NO                                    
1             1                0          1          0   
              2                0          1          0   
              3                0          1          0   
              4                0          1          0   
              5                0          1          0   

                       campaign_type_TypeA_lag4_X_segment_0  \
household_key WEEK_NO                                         
1             1                                         0.0   
              2                                         0.0   
              3                                         0.0   
              4                                         0.0   
              5                                         0.0   

                       campaign_type_TypeA_lag4_X_segment_1  \
household_key WEEK_NO                                         
1             1                                         0.0   
              2                                         0.0   
              3                                         0.0   
              4                                         0.0   
              5                                         0.0   

                       campaign_type_TypeA_lag4_X_segment_2  \
household_key WEEK_NO                                         
1             1                                         0.0   
    

In [49]:
# coefficient extraction
from linearmodels.panel import PanelOLS

sales_model = PanelOLS(
    y_sales,
    X_sales,
    entity_effects=True,
    time_effects=True,
    check_rank = False,
    drop_absorbed=True
)

sales_results = sales_model.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(sales_results.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                  sales   R-squared:                        0.8616
Estimator:                   PanelOLS   R-squared (Between):              0.9491
No. Observations:              265178   R-squared (Within):               0.8655
Date:                Wed, Jul 01 2026   R-squared (Overall):              0.9099
Time:                        20:05:44   Log-likelihood                -1.131e+06
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   1.362e+05
Entities:                        2500   P-value                           0.0000
Avg Obs:                       106.07   Distribution:               F(12,262565)
Min Obs:                       102.00                                           
Max Obs:                       158.00   F-statistic (robust):             2681.1
                            

In [50]:
qty_model = PanelOLS(
    y_qty,
    X_qty,
    entity_effects=True,
    time_effects=True,
    check_rank = False,
    drop_absorbed=True
)


qty_results = qty_model.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(qty_results.summary)



                          PanelOLS Estimation Summary                           
Dep. Variable:               quantity   R-squared:                        0.1559
Estimator:                   PanelOLS   R-squared (Between):              0.6371
No. Observations:              265178   R-squared (Within):               0.1628
Date:                Wed, Jul 01 2026   R-squared (Overall):              0.3415
Time:                        20:06:00   Log-likelihood                -2.523e+06
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      4042.4
Entities:                        2500   P-value                           0.0000
Avg Obs:                       106.07   Distribution:               F(12,262565)
Min Obs:                       102.00                                           
Max Obs:                       158.00   F-statistic (robust):             61.795
                            

In [51]:
#need to change lag for based on qty for qty model.

In [52]:
print(len(sales_results.params.index))
print(sales_results.params.index.tolist())

12
['campaign_type_TypeA_lag4', 'campaign_type_TypeB_lag4', 'campaign_type_TypeC_lag4', 'sales_lag1', 'sales_lag4', 'baskets', 'unique_products', 'campaign_type_TypeA_lag4_X_segment_0', 'campaign_type_TypeA_lag4_X_segment_1', 'campaign_type_TypeB_lag4_X_segment_0', 'campaign_type_TypeB_lag4_X_segment_1', 'campaign_type_TypeC_lag4_X_segment_0']


In [53]:
base_features_sales_nonint = [
  

    "sales_lag1",
    "sales_lag4",

    "baskets",
    "unique_products"
]

X_sales_nonint = panel[
    base_features_sales_nonint + interaction_cols]

In [54]:
from linearmodels.panel import PanelOLS

sales_model = PanelOLS(
    y_sales,
    X_sales_nonint,
    entity_effects=True,
    time_effects=True,
    check_rank = False,
    drop_absorbed=True
)

sales_results = sales_model.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(sales_results.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                  sales   R-squared:                        0.8616
Estimator:                   PanelOLS   R-squared (Between):              0.9491
No. Observations:              265178   R-squared (Within):               0.8655
Date:                Wed, Jul 01 2026   R-squared (Overall):              0.9099
Time:                        20:06:16   Log-likelihood                -1.131e+06
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   1.362e+05
Entities:                        2500   P-value                           0.0000
Avg Obs:                       106.07   Distribution:               F(12,262565)
Min Obs:                       102.00                                           
Max Obs:                       158.00   F-statistic (robust):             2681.1
                            

### **HYPERPARAMETER TUNING**

In [55]:
import optuna
def objective(trial):
    entity_effects = trial.suggest_categorical("entity_effects", [True, False])
    time_effects = trial.suggest_categorical("time_effects", [True, False])

    model = PanelOLS(
        y_sales,
        X_sales,
        entity_effects=entity_effects,
        time_effects=time_effects,
        check_rank = False,
        drop_absorbed=True
    )

    result = model.fit(
        cov_type="clustered",
        cluster_entity=True
        )
    
    return result.rsquared_within
    

In [56]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

[I 2026-07-01 20:06:17,604] A new study created in memory with name: no-name-9108af4c-2ba2-4b2f-8e16-78c98fb7bb82
[I 2026-07-01 20:06:20,260] Trial 0 finished with value: 0.8655009414199983 and parameters: {'entity_effects': True, 'time_effects': False}. Best is trial 0 with value: 0.8655009414199983.
[I 2026-07-01 20:06:22,834] Trial 1 finished with value: 0.8655009414199983 and parameters: {'entity_effects': True, 'time_effects': False}. Best is trial 0 with value: 0.8655009414199983.
[I 2026-07-01 20:06:36,288] Trial 2 finished with value: 0.8654881703837414 and parameters: {'entity_effects': True, 'time_effects': True}. Best is trial 0 with value: 0.8655009414199983.
[I 2026-07-01 20:06:37,644] Trial 3 finished with value: 0.8612104120876996 and parameters: {'entity_effects': False, 'time_effects': False}. Best is trial 0 with value: 0.8655009414199983.
[I 2026-07-01 20:06:51,516] Trial 4 finished with value: 0.8654881703837414 and parameters: {'entity_effects': True, 'time_effects

In [57]:
print(study.best_params)
print(study.best_value)

{'entity_effects': True, 'time_effects': False}
0.8655009414199983


In [58]:
optuna.visualization.plot_optimization_history(study)

In [59]:
optuna.visualization.plot_parallel_coordinate(study)